In [2]:
import os
import re
import pandas as pd
import requests
from dotenv import load_dotenv
from time import sleep

# === Setup ===
env_path = "All_Tokens.env"
load_dotenv(env_path)
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")
token_index = 0

# === Paths ===
input_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output.csv"
output_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step4_ci_detection_output.csv"

# === CI Patterns ===
ci_patterns = {
    r'\.travis\.yml$': 'Travis CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'CircleCI',
    r'\.circleci/config\.yml$': 'CircleCI',
    r'azure-pipelines\.yml$': 'Azure Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}
ci_types = sorted(set(ci_patterns.values()))

# === Load input data ===
if os.path.exists(output_csv):
    df = pd.read_csv(output_csv)
    print("🔁 Resuming from previously saved output.")
else:
    df = pd.read_csv(input_csv)
    df = df[df['Valid_Repo_Step3'].str.lower() == 'yes'].copy()
    df['html_url'] = df['html_url'].astype(str).str.strip()
    df['yml_detected'] = 'none'
    df['total_yml_files'] = 0
    for ci in ci_types:
        df[f"{ci}_count"] = 0

# === Process only unprocessed repos ===
to_process = df[df['yml_detected'] == 'none'].copy()
print(f"🔍 Starting detection: {len(to_process)} repos to review.")

for idx, row in to_process.iterrows():
    url = row['html_url']
    print(f"🔎 [{idx + 1}/{len(df)}] Checking: {url}")
    try:
        parts = url.rstrip('/').split('/')
        owner, repo = parts[-2], parts[-1]

        headers = {'Authorization': f'token {tokens[token_index % len(tokens)]}'}
        token_index += 1

        # === Get default branch ===
        r1 = requests.get(f"https://api.github.com/repos/{owner}/{repo}", headers=headers)
        if r1.status_code != 200:
            print(f"❌ Repo info failed: {r1.status_code}")
            df.at[idx, 'yml_detected'] = 'no'
            continue

        default_branch = r1.json().get('default_branch', 'main')

        # === Get file tree of default branch ===
        r2 = requests.get(f"https://api.github.com/repos/{owner}/{repo}/git/trees/{default_branch}?recursive=1", headers=headers)
        if r2.status_code != 200:
            print(f"❌ File tree failed: {r2.status_code}")
            df.at[idx, 'yml_detected'] = 'no'
            continue

        files = [item['path'] for item in r2.json().get('tree', []) if item['type'] == 'blob']
        matched = []
        for f in files:
            for pattern, ci_type in ci_patterns.items():
                if re.search(pattern, f, re.IGNORECASE):
                    matched.append((f, ci_type))
                    break

        if matched:
            df.at[idx, 'yml_detected'] = 'yes'
        else:
            df.at[idx, 'yml_detected'] = 'no'

        df.at[idx, 'total_yml_files'] = len(matched)

        ci_counts = {}
        for _, ci in matched:
            ci_counts[ci] = ci_counts.get(ci, 0) + 1

        for ci in ci_types:
            df.at[idx, f"{ci}_count"] = ci_counts.get(ci, 0)

        # Save interim result
        df.to_csv(output_csv, index=False)

    except Exception as e:
        print(f"⚠️ Error on {url}: {e}")
        continue

print("\n✅ CI YML scan complete. Output saved.")


🔍 Starting detection: 14690 repos to review.
🔎 [26/14690] Checking: https://github.com/Dawnthorn/nagare
🔎 [33/14690] Checking: https://github.com/bpellin/keepassdroid
🔎 [48/14690] Checking: https://github.com/connectbot/connectbot
🔎 [61/14690] Checking: https://github.com/JakeWharton/SMSMorse
🔎 [62/14690] Checking: https://github.com/JakeWharton/SMSBarrage
🔎 [68/14690] Checking: https://github.com/millenomi/diceshaker


KeyboardInterrupt: 